In [7]:
# import libraries (LangChain 0.3.x + langchain-community)
from langchain.prompts import PromptTemplate
from langchain.chains import RetrievalQA
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Pinecone
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_community.llms import HuggingFacePipeline
from langchain_text_splitters import RecursiveCharacterTextSplitter

import pinecone



In [8]:
PINECONE_API_KEY = "pcsk_R1qAJ_RQ4qhkQfqkVRuFJXugMziJkYeGt2xUj6LaxAPfX4X9Q3X7o4DVhzZ7EgkWtX3rV"


In [ ]:
# --- Session switch ---
# True = reconnect to your existing Pinecone index (fast after a kernel restart — no PDF reload / no upsert).
# False = rebuild from data/: load PDFs, chunk, embed, upsert (~33k vectors; often ~1 hour).
SKIP_EXPENSIVE_INGEST = True


In [9]:
# Extract text from ALL PDFs in a folder (not just one file).
# DirectoryLoader + glob="*.pdf" loads every matching PDF into one list of Documents.

from pathlib import Path


def load_all_pdf_documents(data_dir: str | Path, recursive: bool = False):
    """Load every PDF under `data_dir`.

    Renamed from load_pdfs so a past typo (`load_pdfs = "data/"`) cannot shadow this function in the kernel.

    - recursive=False: only *.pdf in that folder (same as your sample).
    - recursive=True: also PDFs in subfolders (**/*.pdf).
    """
    root = Path(data_dir).resolve()
    pattern = "**/*.pdf" if recursive else "*.pdf"

    loader = DirectoryLoader(
        str(root),
        glob=pattern,
        loader_cls=PyPDFLoader,
        show_progress=True,  # progress bar when loading many files
    )
    return loader.load()


# Example — your repo uses a top-level `data/` folder with many PDFs:
# documents = load_all_pdf_documents("data")
# documents = load_all_pdf_documents("data", recursive=True)  # PDFs in subfolders too



In [10]:
# Call load_all_pdf_documents(...). Old typo load_pdfs=("data/") turned load_pdfs into a string — restart kernel if needed.
import os

if SKIP_EXPENSIVE_INGEST:
    extracted_data = []
else:
    extracted_data = load_all_pdf_documents("data")

    # One LangChain Document per PDF page (typical); many pages ⇒ many rows.
    print(f"Total loaded chunks: {len(extracted_data)}")
    pdf_files = sorted({os.path.basename(d.metadata["source"]) for d in extracted_data})
    print(f"Unique PDF files represented: {len(pdf_files)}")
    print(pdf_files[:20], "..." if len(pdf_files) > 20 else "")



100%|██████████| 231/231 [04:03<00:00,  1.06s/it]

Total loaded chunks: 4454
Unique PDF files represented: 231
['001.pdf', '002.pdf', '1-s2.0-S0263822320332116-main (1).pdf', '1-s2.0-S0263822320332116-main.pdf', '1-s2.0-S0950061800000179-main.pdf', '1-s2.0-S0950061822026745-main.pdf', '1-s2.0-S0950061823034104-main.pdf', '1-s2.0-S2214509522005149-main.pdf', '1-s2.0-S2352012424013109-main.pdf', '1-s2.0-S2352710222011536-main.pdf', '10. A review on the ductility design method of fiber-reinforcedpolymer bar and future prospects.pdf', '10.pdf', '102_sustained load.pdf', '104_hybrid.pdf', '106_hybrid.pdf', '108_hybrid.pdf', '108_sustained load.pdf', '11. Self-monitoring, pseudo-ductile, hybrid FRP reinforcement rods for concrete applications.pdf', '11.pdf', '12. Tensile property analysis of carbon glass hybrid fibers-reinforced graphene-modified polymer bars.pdf'] ...


In [11]:
# Split page-level Documents into smaller overlapping chunks for embeddings / RAG.
# This is NOT the same as loader output: each page may become many chunks.


def text_split(extracted_data):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=20,
    )
    text_chunks = text_splitter.split_documents(extracted_data)
    return text_chunks


In [12]:
if SKIP_EXPENSIVE_INGEST:
    text_chunks = []
else:
    text_chunks = text_split(extracted_data)

    print(f"Documents from PDFs (pages): {len(extracted_data)}")
    print(f"Chunks after RecursiveCharacterTextSplitter: {len(text_chunks)}")



Documents from PDFs (pages): 4454
Chunks after RecursiveCharacterTextSplitter: 33318


In [13]:
# Sentence-transformers model — downloaded/cached from Hugging Face Hub on first use (~90MB).
def download_hugging_face_embeddings(
    model_name: str = "sentence-transformers/all-MiniLM-L6-v2",
):
    """Return a LangChain HuggingFaceEmbeddings instance for chunk vectors."""
    return HuggingFaceEmbeddings(model_name=model_name)


In [14]:
embeddings = download_hugging_face_embeddings()
embeddings

C:\Users\admin\AppData\Local\Temp\ipykernel_41512\2403921603.py:6: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  return HuggingFaceEmbeddings(model_name=model_name)
c:\Users\admin\anaconda3\envs\e2ecivil\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7804.55it/s]


HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({})
), model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False, show_progress=False)

In [15]:
# Smoke test: embed a short string — vector dimension matches the model (MiniLM-L6-v2 → 384).
query_result = embeddings.embed_query("Hello world")
print("Length", len(query_result))


Length 384


In [16]:
# Pinecone (SDK v3+): no pinecone.init(), no "environment" string — only PINECONE_API_KEY.
# Create an index in the Pinecone console first: dimension 384, metric cosine (matches all-MiniLM-L6-v2).
# LangChain's helper reads the API key from the OS environment variable PINECONE_API_KEY.

import os

os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY

index_name = "chatbot"  # must match an existing index name in your project


In [18]:
# Embed each chunk and upsert into Pinecone (tutorial-style).
# Full ~33k chunks takes a long time — set TEST_LIMIT to a small number first.
# LangChain's from_texts() uses a 30s HTTP timeout; slow/VPN networks can hit SSL handshake timeouts.
# Here we build the client with a longer timeout, then add_texts (same end result as from_texts).

TEST_LIMIT = None  # e.g. 200 for a dry run; None = upload all chunks_to_upload

from pinecone import Pinecone as PineconeSdk

PINECONE_TIMEOUT_SEC = 180  # increase (e.g. 300) if PineconeTimeoutError persists

if SKIP_EXPENSIVE_INGEST:
    # Index already populated — reconnect only (no duplicate upsert).
    vectorstore = Pinecone.from_existing_index(
        index_name=index_name,
        embedding=embeddings,
        text_key="text",
    )
else:
    chunks_to_upload = text_chunks if TEST_LIMIT is None else text_chunks[:TEST_LIMIT]

    _pc = PineconeSdk(
        api_key=os.environ["PINECONE_API_KEY"],
        timeout=PINECONE_TIMEOUT_SEC,
        pool_threads=8,
    )
    _pinecone_index = _pc.Index(index_name)

    vectorstore = Pinecone(_pinecone_index, embeddings, text_key="text")
    vectorstore.add_texts(
        [t.page_content for t in chunks_to_upload],
        metadatas=[dict(t.metadata) for t in chunks_to_upload],
    )

vectorstore



In [19]:
vectorstore.similarity_search("resin type", k=10)

[Document(metadata={'page': 106, 'source': 'C:\\Users\\admin\\e2e_civil\\data\\2003_David.pdf'}, page_content='Resin \nFiber'),
 Document(metadata={'page': 106, 'source': 'C:\\Users\\admin\\e2e_civil\\data\\2003_David.pdf'}, page_content='Resin \nFiber'),
 Document(metadata={'page': 106, 'source': 'C:\\Users\\admin\\e2e_civil\\data\\2003_David.pdf'}, page_content='Resin \nFiber'),
 Document(metadata={'page': 106, 'source': 'C:\\Users\\admin\\e2e_civil\\data\\2003_David.pdf'}, page_content='Resin \nFiber'),
 Document(metadata={'page': 106, 'source': 'C:\\Users\\admin\\e2e_civil\\data\\2003_David.pdf'}, page_content='Resin \nFiber'),
 Document(metadata={'page': 106, 'source': 'C:\\Users\\admin\\e2e_civil\\data\\2003_David.pdf'}, page_content='Resin \nFiber'),
 Document(metadata={'page': 106, 'source': 'C:\\Users\\admin\\e2e_civil\\data\\2003_David.pdf'}, page_content='Resin \nFiber'),
 Document(metadata={'page': 106, 'source': 'C:\\Users\\admin\\e2e_civil\\data\\2003_David.pdf'}, page_co

In [20]:
# RAG prompt (matches common tutorial pattern): {context} + {question} → one "Helpful answer".
prompt_template = """
Use the following pieces of information to answer the user's question.
If you don't know the answer, just say that you don't know, don't try to make up an answer.

Context: {context}
Question: {question}

Only return the helpful answer below and nothing else.
Helpful answer:
"""


In [22]:
PROMPT = PromptTemplate(template=prompt_template, input_variables=["context", "question"])
chain_type_kwargs = {"prompt": PROMPT}

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})


In [23]:
# --- Local LLM: GPU → your Llama 3.2 3B on disk; CPU → small Hub model (avoids OOM). ---
# Files in model/ are on disk. Each kernel session loads into RAM/VRAM to run — not re-downloading Llama.
import torch
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

LOCAL_LLAMA_3B = Path("model") / "Llama-3.2-3B-Instruct"
CPU_LIGHT_MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

use_cuda = torch.cuda.is_available()
if use_cuda:
    model_id = str(LOCAL_LLAMA_3B)
    _dtype = torch.float16
    _load_kw = {"torch_dtype": _dtype, "device_map": "auto"}
else:
    model_id = CPU_LIGHT_MODEL_ID
    _dtype = torch.float32
    _load_kw = {"torch_dtype": _dtype, "low_cpu_mem_usage": True}

print(
    "LLM:",
    "CUDA → local Llama 3.2 3B" if use_cuda else "CPU → lightweight " + CPU_LIGHT_MODEL_ID,
)

tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    trust_remote_code=True,
    clean_up_tokenization_spaces=False,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

hf_model = AutoModelForCausalLM.from_pretrained(model_id, trust_remote_code=True, **_load_kw)
if not use_cuda:
    hf_model = hf_model.to("cpu")

# Safer defaults for small Qwen + RAG (reduces rambling / numbered junk)
pipe = pipeline(
    "text-generation",
    model=hf_model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    do_sample=True,
    temperature=0.2,
    top_p=0.9,
    repetition_penalty=1.15,
    return_full_text=False,
    pad_token_id=tokenizer.eos_token_id,
)
llm = HuggingFacePipeline(pipeline=pipe)

# Tutorial-style chain (docsearch → vectorstore; k=2 like the video)
qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vectorstore.as_retriever(search_kwargs={"k": 5}),
    return_source_documents=True,
    chain_type_kwargs=chain_type_kwargs,
)


LLM: CPU → lightweight Qwen/Qwen2.5-0.5B-Instruct


Loading weights: 100%|██████████| 290/290 [00:01<00:00, 187.23it/s]


In [ ]:
# REMOVED: the old `while True: input()` loop blocked the entire kernel and forced Interrupt/restarts.
# Ask questions below: set QUERY = "..." and run that cell. Re-run only that cell for each new question.
# With SKIP_EXPENSIVE_INGEST = True you can restart the kernel and skip the hour-long ingest — see the flag cell above.
pass



### Where to type your question

Edit **`QUERY = "..."`** in the code cell below and **Run** that cell. Run it again after each edit for a new question.

**After a kernel restart:** keep **`SKIP_EXPENSIVE_INGEST = True`** (cell below the API key), then **Run All**. You will **not** reload PDFs or re-upsert chunks; only embeddings load, Pinecone reconnects, and the LLM loads.

Set **`SKIP_EXPENSIVE_INGEST = False`** only when you intentionally rebuild the index from `data/`.

There is **no** `input()` loop — it blocked other cells and caused endless restarts.


In [ ]:
# Your question goes in QUERY below — inside the quote marks. Then Run this cell.
QUERY = "what are different resin types"

_r = qa.invoke({"query": QUERY})
print("Response:", _r["result"])


In [ ]:
# --- Quick check: see the filled prompt (no LLM required) ---
_docs = vectorstore.similarity_search(
    "what are the different types of resins?", k=4
)
_context = "\n\n".join(d.page_content for d in _docs)
_example_q = "What are the different types of frp bars?"
print(PROMPT.format(context=_context[:2000], question=_example_q)[:2500])
print("...")



Use the following pieces of information to answer the user's question.
If you don't know the answer, just say that you don't know, don't try to make up an answer.

Context: speciﬁcations and design guidelines (ACI 440.6M [3]; CAN/CSA
S807 [32]) have also been developed to encourage the construction
industry to use FRP bars. This has resulted in many demonstration
projects and ﬁeld applications, such as bridges [27], parking ga-
rages [25], water-treatment plants [47], bridge barriers [36], con-
crete pavement [26], and jetties [42].
Different types of ﬁbers are used in manufacturing FRP bars such
as carbon, glass, aramid, and basalt. Many studies have been carried

speciﬁcations and design guidelines (ACI 440.6M [3]; CAN/CSA
S807 [32]) have also been developed to encourage the construction
industry to use FRP bars. This has resulted in many demonstration
projects and ﬁeld applications, such as bridges [27], parking ga-
rages [25], water-treatment plants [47], bridge barriers [36], con